In [0]:
CREATE OR REPLACE TEMPORARY VIEW MPSII_TREATMENT_TABLE AS
SELECT *
FROM (
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME,
        PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
) t
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';

In [0]:
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified_without_filter AS 
(
  SELECT * FROM (
    -- Medical Events
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      PLACE_OF_SERVICE,
      BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS HCP_NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL AS PLACE_OF_SERVICE,
      PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
  ) AS combined
);

SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT HCP_NPI)    AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_1Dx_Specified_without_filter;


In [0]:
-- Dx ∩ Tx cohort counts (patients with MPS II Dx + at least one MPS II treatment claim)

WITH dx_patients AS (
  SELECT DISTINCT PATIENT_ID
  FROM MPSII_1Dx_Specified_without_filter
),
tx_dx AS (
  SELECT t.*
  FROM MPSII_TREATMENT_TABLE t
  WHERE t.PATIENT_ID IN (SELECT PATIENT_ID FROM dx_patients)
)
SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT NPI)        AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM tx_dx;


In [0]:
-- 1Dx base (NO date filter, NO PAID filter)
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified AS 
(
  SELECT * FROM (
    -- Medical Events
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      PLACE_OF_SERVICE,
      BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS HCP_NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL AS PLACE_OF_SERVICE,
      PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
  ) AS combined
);

-- 2Dx patient universe as a TEMP VIEW (>=2 distinct FILL_DATE)
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Specified_without_filter AS
(
  SELECT DISTINCT PATIENT_ID
  FROM (
    SELECT
      PATIENT_ID,
      COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
    FROM MPSII_1Dx_Specified
    GROUP BY PATIENT_ID
  )
  WHERE NUMBER_OF_CLAIMS >= 2
);

-- Counts restricted to 2Dx patients
SELECT
  COUNT(DISTINCT a.PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT a.HCP_NPI)    AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT a.HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_1Dx_Specified a
JOIN MPSII_2Dx_Specified_without_filter b
  ON a.PATIENT_ID = b.PATIENT_ID;


In [0]:
WITH MPSII_2Dx_Tx_Specified_Tx_claims AS (
  SELECT *
  FROM MPSII_TREATMENT_TABLE
  WHERE PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM MPSII_2Dx_Specified_without_filter)
)
SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT NPI)        AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_2Dx_Tx_Specified_Tx_claims;


In [0]:
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified_20200801_20251130 AS
(
  SELECT * FROM (
    -- Medical Events
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      PLACE_OF_SERVICE,
      BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS HCP_NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL AS PLACE_OF_SERVICE,
      PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
);

SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT HCP_NPI)    AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_1Dx_Specified_20200801_20251130;


In [0]:
-- Treatment claims for patients with MPS II Dx (within 2020-08-01 to 2025-11-30)
WITH MPSII_1Dx_Tx_Claims AS (
  SELECT *
  FROM MPSII_TREATMENT_TABLE t
  WHERE t.PATIENT_ID IN (
    SELECT DISTINCT PATIENT_ID
    FROM MPSII_1Dx_Specified_20200801_20251130
  )
)
SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT NPI)        AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_1Dx_Tx_Claims;


In [0]:
-- 1Dx base with date window
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified_20200801_20251130 AS
(
  SELECT * FROM (
    -- Medical Events
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      PLACE_OF_SERVICE,
      BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS HCP_NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL AS PLACE_OF_SERVICE,
      PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
);

-- 2Dx patient universe (>=2 distinct FILL_DATE) within the above 1Dx view
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Specified_20200801_20251130 AS
(
  SELECT DISTINCT PATIENT_ID
  FROM (
    SELECT
      PATIENT_ID,
      COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
    FROM MPSII_1Dx_Specified_20200801_20251130
    GROUP BY PATIENT_ID
  )
  WHERE NUMBER_OF_CLAIMS >= 2
);

-- Counts restricted to 2Dx patients
SELECT
  COUNT(DISTINCT a.PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT a.HCP_NPI)    AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT a.HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_1Dx_Specified_20200801_20251130 a
JOIN MPSII_2Dx_Specified_20200801_20251130 b
  ON a.PATIENT_ID = b.PATIENT_ID;


In [0]:
WITH MPSII_2Dx_Tx_Claims_20200801_20251130 AS (
  SELECT *
  FROM MPSII_TREATMENT_TABLE t
  WHERE t.PATIENT_ID IN (
    SELECT DISTINCT PATIENT_ID
    FROM MPSII_2Dx_Specified_20200801_20251130
  )
)
SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT NPI)        AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_2Dx_Tx_Claims_20200801_20251130;


In [0]:
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified_PAID_20200801_20251130 AS
(
  SELECT * FROM (
    -- Medical Events
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      PLACE_OF_SERVICE,
      BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS HCP_NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL AS PLACE_OF_SERVICE,
      PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
);

SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT HCP_NPI)    AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_1Dx_Specified_PAID_20200801_20251130;


In [0]:
WITH MPSII_1Dx_PAID_Tx_Claims_20200801_20251130 AS (
  SELECT *
  FROM MPSII_TREATMENT_TABLE t
  WHERE t.PATIENT_ID IN (
    SELECT DISTINCT PATIENT_ID
    FROM MPSII_1Dx_Specified_PAID_20200801_20251130
  )
)
SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT NPI)        AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_1Dx_PAID_Tx_Claims_20200801_20251130;


In [0]:
-- 1Dx base with PAID filter (pharmacy) + date window
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified_PAID_20200801_20251130 AS
(
  SELECT * FROM (
    -- Medical Events
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      PLACE_OF_SERVICE,
      BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS HCP_NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL AS PLACE_OF_SERVICE,
      PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
);

-- 2Dx patient universe (>=2 distinct FILL_DATE) within the above 1Dx view
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Specified_PAID_20200801_20251130 AS
(
  SELECT DISTINCT PATIENT_ID
  FROM (
    SELECT
      PATIENT_ID,
      COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
    FROM MPSII_1Dx_Specified_PAID_20200801_20251130
    GROUP BY PATIENT_ID
  )
  WHERE NUMBER_OF_CLAIMS >= 2
);

-- Counts restricted to 2Dx patients
SELECT
  COUNT(DISTINCT a.PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT a.HCP_NPI)    AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT a.HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_1Dx_Specified_PAID_20200801_20251130 a
JOIN MPSII_2Dx_Specified_PAID_20200801_20251130 b
  ON a.PATIENT_ID = b.PATIENT_ID;


In [0]:
WITH MPSII_2Dx_PAID_Tx_Claims_20200801_20251130 AS (
  SELECT *
  FROM MPSII_TREATMENT_TABLE t
  WHERE t.PATIENT_ID IN (
    SELECT DISTINCT PATIENT_ID
    FROM MPSII_2Dx_Specified_PAID_20200801_20251130
  )
)
SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT NPI)        AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_2Dx_PAID_Tx_Claims_20200801_20251130;


In [0]:
CREATE OR REPLACE TEMPORARY VIEW MPSII_TREATMENT_TABLE_without_filter AS
SELECT *
FROM (
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME,
        PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
) t;

SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT NPI)        AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_TREATMENT_TABLE_without_filter;


In [0]:
CREATE OR REPLACE TEMPORARY VIEW MPSII_TREATMENT_TABLE_20230801_20251130 AS
SELECT *
FROM (
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME,
        PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
) t
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';

SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT NPI)        AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_TREATMENT_TABLE_20230801_20251130;


In [0]:
CREATE OR REPLACE TEMPORARY VIEW MPSII_TREATMENT_TABLE_PAID_20230801_20251130 AS
SELECT *
FROM (
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME,
        PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
) t
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';

SELECT
  COUNT(DISTINCT PATIENT_ID) AS cnt_distinct_patient_id,
  COUNT(DISTINCT NPI)        AS cnt_distinct_hcp_npi,
  COUNT(DISTINCT HCO_NPI)    AS cnt_distinct_hco_npi
FROM MPSII_TREATMENT_TABLE_PAID_20230801_20251130;
